<a href="https://colab.research.google.com/github/replysantosh-lang/ECARAgenticAI/blob/main/09_L3_Reflection_and_Blogpost_Writing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Lesson 3: Reflection and Blogpost Writing

## Setup

In [18]:
from google.colab import userdata

llm_config = {"model": "gpt-3.5-turbo", "api_key": userdata.get('OPENAI_API_KEY')}

## The task!

In [19]:
task = '''
        Write a concise but engaging blogpost about
       DeepLearning.AI. Make sure the blogpost is
       within 100 words.
       '''


## Create a writer agent

In [20]:
!pip install autogen
import autogen

writer = autogen.AssistantAgent(
    name="Writer",
    system_message="You are a writer. You write engaging and concise "
        "blogpost (with title) on given topics. You must polish your "
        "writing based on the feedback you receive and give a refined "
        "version. Only return your final work without additional comments.",
    llm_config=llm_config,
)

In [21]:
reply = writer.generate_reply(messages=[{"content": task, "role": "user"}])

In [22]:
print(reply)

Title: Unleashing the Power of Deep Learning with DeepLearning.AI

Dive into the world of deep learning with DeepLearning.AI! Whether you're a beginner or an expert, this platform offers top-notch courses to help you master the art of AI. With engaging lessons, real-world projects, and expert instructors like Andrew Ng, you'll unlock the potential of deep learning. Elevate your skills, advance your career, and join a community of learners passionate about AI. Don't miss this opportunity to dive deep into the realms of artificial intelligence with DeepLearning.AI!


## Adding reflection

Create a critic agent to reflect on the work of the writer agent.

In [23]:
critic = autogen.AssistantAgent(
    name="Critic",
    is_termination_msg=lambda x: x.get("content", "").find("TERMINATE") >= 0,
    llm_config=llm_config,
    system_message="You are a critic. You review the work of "
                "the writer and provide constructive "
                "feedback to help improve the quality of the content.",
)

In [24]:
res = critic.initiate_chat(
    recipient=writer,
    message=task,
    max_turns=2,
    summary_method="last_msg"
)

Critic (to Writer):


        Write a concise but engaging blogpost about
       DeepLearning.AI. Make sure the blogpost is
       within 100 words.
       

--------------------------------------------------------------------------------
Writer (to Critic):

Title: "Unlocking the Power of AI with DeepLearning.AI"

Dive into the dynamic world of artificial intelligence with DeepLearning.AI! Led by industry expert Andrew Ng, this innovative platform offers cutting-edge courses on deep learning and AI technology. Whether you're a seasoned professional or a curious beginner, DeepLearning.AI provides accessible and practical knowledge to help you thrive in this rapidly evolving field. From neural networks to computer vision, this platform equips you with the skills and tools necessary to transform your career. Join us on this exciting journey towards unlocking the full potential of AI with DeepLearning.AI!

--------------------------------------------------------------------------------
Cr

## Nested chat

In [27]:
SEO_reviewer = autogen.AssistantAgent(
    name="SEO_Reviewer",
    llm_config=llm_config,
    system_message="You are an SEO reviewer, known for "
        "your ability to optimize content for search engines, "
        "ensuring that it ranks well and attracts organic traffic. "
        "Make sure your suggestion is concise (within 3 bullet points), "
        "concrete and to the point. "
        "Begin the review by stating your role.",
)

In [29]:
legal_reviewer = autogen.AssistantAgent(
    name="Legal_Reviewer",
    llm_config=llm_config,
    system_message="You are a legal reviewer, known for "
        "your ability to ensure that content is legally compliant "
        "and free from any potential legal issues. "
        "Make sure your suggestion is concise (within 3 bullet points), "
        "concrete and to the point. "
        "Begin the review by stating your role.",
)

In [33]:
ethics_reviewer = autogen.AssistantAgent(
    name="Ethics_Reviewer",
    llm_config=llm_config,
    system_message="You are an ethics reviewer, known for "
        "your ability to ensure that content is ethically sound "
        "and free from any potential ethical issues. "
        "Make sure your suggestion is concise (within 3 bullet points), "
        "concrete and to the point. "
        "Begin the review by stating your role. ",
)

In [34]:
meta_reviewer = autogen.AssistantAgent(
    name="Meta_Reviewer",
    llm_config=llm_config,
    system_message="You are a meta reviewer, you aggragate and review "
    "the work of other reviewers and give a final suggestion on the content.",
)

## Orchestrate the nested chats to solve the task

In [35]:
def reflection_message(recipient, messages, sender, config):
    return f'''Review the following content.
            \n\n {recipient.chat_messages_for_summary(sender)[-1]['content']}'''

review_chats = [
    {
     "recipient": SEO_reviewer,
     "message": reflection_message,
     "summary_method": "reflection_with_llm",
     "summary_args": {"summary_prompt" :
        "Return review into as JSON object only:"
        "{'Reviewer': '', 'Review': ''}. Here Reviewer should be your role",},
     "max_turns": 1},
    {
    "recipient": legal_reviewer, "message": reflection_message,
     "summary_method": "reflection_with_llm",
     "summary_args": {"summary_prompt" :
        "Return review into as JSON object only:"
        "{'Reviewer': '', 'Review': ''}.",},
     "max_turns": 1},
    {"recipient": ethics_reviewer, "message": reflection_message,
     "summary_method": "reflection_with_llm",
     "summary_args": {"summary_prompt" :
        "Return review into as JSON object only:"
        "{'reviewer': '', 'review': ''}",},
     "max_turns": 1},
     {"recipient": meta_reviewer,
      "message": "Aggregrate feedback from all reviewers and give final suggestions on the writing.",
     "max_turns": 1},
]


In [36]:
critic.register_nested_chats(
    review_chats,
    trigger=writer,
)

**Note**: You might get a slightly different response than what's shown in the video. Feel free to try different task.

In [37]:
res = critic.initiate_chat(
    recipient=writer,
    message=task,
    max_turns=2,
    summary_method="last_msg"
)

Critic (to Writer):


        Write a concise but engaging blogpost about
       DeepLearning.AI. Make sure the blogpost is
       within 100 words.
       

--------------------------------------------------------------------------------
Writer (to Critic):

Title: "Master Deep Learning with DeepLearning.AI"

Are you looking to delve into the world of artificial intelligence and deep learning? Look no further than DeepLearning.AI! Offering comprehensive courses created by industry expert Andrew Ng, DeepLearning.AI provides the perfect opportunity to enhance your skills and stay ahead in this rapidly evolving field. From deep learning specialization to natural language processing, their courses cover it all. Join the millions of learners worldwide who have benefited from DeepLearning.AI's top-notch resources and take your AI knowledge to the next level. Don't miss this chance to master deep learning and shape the future of technology!

--------------------------------------------------

## Get the summary

In [38]:
print(res.summary)

Title: "Empower Your AI Journey with DeepLearning.AI"

Embark on an enriching learning adventure with DeepLearning.AI, spearheaded by renowned expert Andrew Ng. Explore a diverse array of courses, from deep learning specializations to natural language processing, designed to catapult your AI proficiency to new heights. Join a global community of learners benefitting from top-tier resources and cutting-edge insights. Take charge of your future in technology and shape the innovations of tomorrow. To kickstart your AI mastery, seize this opportunity today with DeepLearning.AI. Let's revolutionize the world of artificial intelligence together!
